# 11. 범용 이미지 수집 (Naver Shopping API → staging)

**사용법**: Cell 1의 `CLASS_NAME` 값만 바꾸고 전체 실행.

**저장 위치**: `data/staging/{CLASS_NAME}/{query_slug}/stg_NNNN.jpg`

**다음 단계**: 수집 완료 후 `08_approve_staging.ipynb` 에서 `CLASS_NAME` 동일하게 설정 → 검수 → approved 이동

**수집 목표**: 클래스당 100장 수집 / 최소 50장 approved

**지원 클래스** (`config.NAVER_SEARCH_QUERIES` 키 기준):
```
refrigerator  washer_dryer    wash_tower
rice_cooker   microwave       air_fryer       electric_kettle
vacuum_cleaner robot_vacuum
fan           air_conditioner heater
dehumidifier  humidifier
monitor       keyboard        mouse
beam_projector
```

**검수 기준** (staged → approved 대상):
- 제품 전체가 찍힌 이미지 (partial crop 제외)
- 단일 제품 정면/측면/비스듬 뷰
- 배경 있어도 무관, 단 제품이 주체

**제외 기준**:
- 여러 제품 동시 노출 (세트 사진)
- 광고 배너, 텍스트 도배
- 사람이 주인공인 이미지
- 다른 클래스 제품이 메인

In [ ]:
import os, sys, re, requests
from io import BytesIO
from pathlib import Path
from PIL import Image as PILImage
from datetime import datetime
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
import config

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

NAVER_ID     = os.getenv('NAVER_CLIENT_ID', '')
NAVER_SECRET = os.getenv('NAVER_CLIENT_SECRET', '')
if not NAVER_ID or not NAVER_SECRET:
    raise EnvironmentError('NAVER_CLIENT_ID / NAVER_CLIENT_SECRET 없음. .env 확인.')
print('Naver API 자격증명 확인 OK')

# ────────────────────────────────────────────────────────────────────────────────
# ★ 여기만 변경 ★
CLASS_NAME = 'refrigerator'
# ────────────────────────────────────────────────────────────────────────────────

IMAGES_PER_QUERY = 100   # 쿼리당 최대 수집 수 (Naver API 최대값)
MIN_IMG_SIZE     = 150   # 이 픽셀 미만 이미지는 저장 안 함

if CLASS_NAME not in config.NAVER_SEARCH_QUERIES:
    raise ValueError(f'{CLASS_NAME} 이 config.NAVER_SEARCH_QUERIES 에 없음.\n'
                     f'사용 가능: {list(config.NAVER_SEARCH_QUERIES.keys())}')

QUERIES      = config.NAVER_SEARCH_QUERIES[CLASS_NAME]
STAGING_DIR  = os.path.join('data', 'staging', CLASS_NAME)
METADATA_CSV = os.path.join(config.METADATA_DIR, f'{CLASS_NAME}_staging_metadata.csv')

print(f'클래스   : {CLASS_NAME}')
print(f'쿼리 수  : {len(QUERIES)}개  |  쿼리당 최대 {IMAGES_PER_QUERY}장  |  총 최대 {len(QUERIES)*IMAGES_PER_QUERY}장')
print(f'staging  : {STAGING_DIR}')
print(f'metadata : {METADATA_CSV}')
print()
for i, q in enumerate(QUERIES):
    print(f'  {i+1:2d}. {q}')

In [ ]:
# ── 수집 함수 정의 ────────────────────────────────────────────────────────────────

def fetch_items(query, display):
    r = requests.get(
        'https://openapi.naver.com/v1/search/shop.json',
        headers={'X-Naver-Client-Id': NAVER_ID, 'X-Naver-Client-Secret': NAVER_SECRET},
        params={'query': query, 'display': display, 'start': 1},
        timeout=config.DOWNLOAD_TIMEOUT,
    )
    r.raise_for_status()
    return r.json().get('items', [])


def download_and_validate(url, save_path, min_px):
    try:
        r = requests.get(url, timeout=config.DOWNLOAD_TIMEOUT)
        if r.status_code != 200 or 'image' not in r.headers.get('Content-Type', ''):
            return False, 0, 0
        img = PILImage.open(BytesIO(r.content)).convert('RGB')
        w, h = img.size
        if w < min_px or h < min_px:
            return False, w, h
        img.save(save_path, 'JPEG', quality=95)
        return True, w, h
    except Exception:
        return False, 0, 0


# ── 수집 실행 ─────────────────────────────────────────────────────────────────────

records = []
n_saved = n_skip = n_exist = 0

os.makedirs(config.METADATA_DIR, exist_ok=True)

for q_i, query in enumerate(QUERIES, 1):
    slug = re.sub(r'[^\w가-힣]', '_', query)
    qdir = os.path.join(STAGING_DIR, slug)
    os.makedirs(qdir, exist_ok=True)

    try:
        items = fetch_items(query, IMAGES_PER_QUERY)
    except Exception as e:
        print(f'  [{q_i:2d}/{len(QUERIES)}] {query}: API 오류 -> {e}')
        continue

    saved_q = exist_q = skip_q = 0

    for idx, item in enumerate(items):
        img_url = item.get('image', '')
        if not img_url:
            skip_q += 1
            continue
        fname       = f'stg_{idx:04d}.jpg'
        save_path   = os.path.join(qdir, fname)
        title_clean = re.sub(r'<[^>]+>', '', item.get('title', ''))

        if os.path.exists(save_path):
            try:
                img = PILImage.open(save_path)
                w, h = img.size
            except Exception:
                w, h = 0, 0
            records.append({'query': query, 'title': title_clean[:120],
                'link': item.get('link', ''), 'image_url': img_url,
                'saved_path': save_path, 'width': w, 'height': h,
                'status': 'staged',
                'collected_at': datetime.now().isoformat(timespec='seconds')})
            exist_q += 1; n_exist += 1
            continue

        ok, w, h = download_and_validate(img_url, save_path, MIN_IMG_SIZE)
        if ok:
            records.append({'query': query, 'title': title_clean[:120],
                'link': item.get('link', ''), 'image_url': img_url,
                'saved_path': save_path, 'width': w, 'height': h,
                'status': 'staged',
                'collected_at': datetime.now().isoformat(timespec='seconds')})
            saved_q += 1; n_saved += 1
        else:
            skip_q += 1; n_skip += 1

    print(f'  [{q_i:2d}/{len(QUERIES)}] {query:30s} -> 저장 {saved_q:3d}장, 기존 {exist_q:3d}장, 스킵 {skip_q:3d}장')

print(f'\n수집 완료 -- 신규 {n_saved}장 | 기존유지 {n_exist}장 | 스킵(소형/오류) {n_skip}장')

In [ ]:
# ── 메타데이터 저장 + staging 현황 출력 ──────────────────────────────────────────

new_df = pd.DataFrame(records)
if os.path.exists(METADATA_CSV):
    old_df = pd.read_csv(METADATA_CSV, encoding='utf-8-sig')
    combined = pd.concat([old_df, new_df], ignore_index=True)
    combined.drop_duplicates(subset=['image_url'], keep='last', inplace=True)
else:
    combined = new_df

combined.to_csv(METADATA_CSV, index=False, encoding='utf-8-sig')
print(f'메타데이터 저장: {METADATA_CSV}  ({len(combined)}행)')

print()
print('=== staging 폴더 현황 ===')
total_staged = 0
for slug in sorted(os.listdir(STAGING_DIR)):
    d = os.path.join(STAGING_DIR, slug)
    if not os.path.isdir(d):
        continue
    imgs = [f for f in os.listdir(d) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f'  {slug:40s}: {len(imgs):3d}장')
    total_staged += len(imgs)
print(f'  {"합계":40s}: {total_staged:3d}장')
print()
print(f'목표 대비: {total_staged}/{len(QUERIES)*IMAGES_PER_QUERY}장 수집됨')

In [ ]:
# ── 검수용 HTML 생성 ──────────────────────────────────────────────────────────────
# 브라우저에서 열어 이미지 확인 후 08_approve_staging.ipynb 에서 APPROVED 목록 작성

import base64

HTML_OUT = os.path.join(config.METADATA_DIR, f'{CLASS_NAME}_staging_review.html')

def img_b64(fpath, px=220):
    try:
        img = PILImage.open(str(fpath)).convert('RGB')
        img.thumbnail((px, px), PILImage.LANCZOS)
        buf = BytesIO()
        img.save(buf, 'JPEG', quality=82)
        return base64.b64encode(buf.getvalue()).decode()
    except Exception:
        return ''

staging_root = Path(STAGING_DIR)
by_q = {}
for d in sorted(staging_root.iterdir()):
    if d.is_dir():
        fs = sorted(d.glob('*.jpg')) + sorted(d.glob('*.jpeg')) + sorted(d.glob('*.png'))
        if fs:
            by_q[d.name] = list(fs)

total_img = sum(len(v) for v in by_q.values())
print(f'HTML 생성 중.. 총 {total_img}장')

CSS = (
    '<style>'
    '*{box-sizing:border-box;margin:0;padding:0}'
    'body{font-family:sans-serif;background:#0f0f0f;color:#ccc;padding:16px}'
    'h1{color:#fff;margin-bottom:4px;font-size:22px}'
    '.sub{color:#888;font-size:12px;margin-bottom:14px}'
    '.warn{background:#2a1010;border:2px solid #c0392b;border-radius:6px;'
    'padding:12px 16px;margin-bottom:18px;font-size:13px;line-height:1.9}'
    '.warn b{color:#e74c3c}'
    '.qh{background:#1a2a3a;color:#7ec8e3;font-size:13px;font-weight:bold;'
    'padding:7px 12px;border-radius:6px 6px 0 0;'
    'border-left:4px solid #7ec8e3;margin-top:18px}'
    '.grid{display:flex;flex-wrap:wrap;gap:5px;padding:8px;'
    'background:#161616;border-radius:0 0 6px 6px}'
    '.card{width:140px;background:#1e1e1e;border-radius:3px;'
    'overflow:hidden;border:1px solid #2a2a2a;position:relative}'
    '.card:hover{border-color:#7ec8e3}'
    '.card img{width:140px;height:120px;object-fit:contain;'
    'background:#0a0a0a;display:block}'
    '.noimg{width:140px;height:120px;display:flex;align-items:center;'
    'justify-content:center;color:#444;font-size:11px}'
    '.idx{position:absolute;top:2px;left:2px;background:rgba(0,0,0,.75);'
    'color:#fff;padding:1px 5px;border-radius:3px;font-size:10px}'
    '.m{padding:4px;font-size:8px;color:#555;word-break:break-all;line-height:1.4}'
    '</style>'
)

WARN = (
    '<div class="warn">'
    '<b>검수 기준 — staging 폴더에서 직접 파일 경로 복사</b><br>'
    '✅ 제품 전체가 찍힌 이미지 (정면·측면·비스듬 OK)<br>'
    '✅ 단일 제품, 배경 있어도 무관<br>'
    '❌ 여러 제품 동시 노출 (세트 사진)<br>'
    '❌ 광고 배너 / 텍스트 도배 / 사람이 주체<br>'
    '❌ 다른 클래스 제품이 메인<br>'
    '❌ 상품 부분만 잘린 이미지 (partial crop)<br>'
    '<br>이미지 경로를 <b>08_approve_staging.ipynb → APPROVED 리스트</b>에 붙여넣으세요.'
    '</div>'
)

parts = []
global_i = 0
for slug, files in by_q.items():
    cards = []
    for fp in files:
        b   = img_b64(fp)
        rel = str(fp).replace('\\', '/')
        img_tag = (f'<img src="data:image/jpeg;base64,{b}" loading="lazy">'
                   if b else '<div class="noimg">ERR</div>')
        cards.append(
            f'<div class="card">'
            f'<div class="idx">#{global_i}</div>'
            f'{img_tag}'
            f'<div class="m">{fp.name}<br>'
            f'<span style="color:#333">{slug}</span></div>'
            f'</div>'
        )
        global_i += 1
    parts.append(
        f'<div class="qh">{slug} <span style="color:#888;font-size:11px">({len(files)}장)</span></div>'
        f'<div class="grid">{"<br>".join(cards).replace("<br>", "")}</div>'
    )

html = (
    '<!DOCTYPE html><html><head><meta charset="utf-8">'
    f'<title>{CLASS_NAME} staging review</title>'
    + CSS + '</head><body>'
    f'<h1>{CLASS_NAME} Staging 검수</h1>'
    f'<div class="sub">총 {total_img}장 | Naver Shopping API | '
    f'approved 이후 08_approve_staging.ipynb 실행</div>'
    + WARN + ''.join(parts) + '</body></html>'
)

with open(HTML_OUT, 'w', encoding='utf-8') as fh:
    fh.write(html)
print(f'[OK] {HTML_OUT}  ({os.path.getsize(HTML_OUT) // 1024} KB)')
print()
print(f'=== 다음 단계 ===')
print(f'1. {HTML_OUT} 를 브라우저에서 열어 이미지 검수')
print(f'2. 08_approve_staging.ipynb 열기')
print(f'3. Cell 1 에서 CLASS_NAME = \'{CLASS_NAME}\' 로 설정')
print(f'4. Cell 5 의 APPROVED 리스트에 승인할 파일 경로 입력')
print(f'5. Cell 6 실행 → data/processed/{CLASS_NAME}/ 로 이동')